In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib import cm
import pandas as pd
from PIL import Image
import plotly.graph_objects as go
from itertools import product
from scipy.stats import multivariate_normal

In [2]:
def compute_class_probs(z, means, kappa, M, beta=0.5):
    K = means.shape[0]
    distance = np.array(
        [
            np.linalg.norm(z[np.newaxis, i] - means, axis=1) ** 2
            for i in range(z.shape[0])
        ]
    )
    p_z_c = ((1 - beta) / K) * (kappa / (2 * np.pi)) * np.exp(-kappa * distance)
    p_z_oog = np.array([beta / (M**2)])
    p_z_oog = np.broadcast_to(p_z_oog, (p_z_c.shape[0], 1))
    p_z_c = np.concatenate([p_z_c, p_z_oog], axis=1)
    p_z = np.sum(p_z_c, axis=1)
    p_c_z = p_z_c / p_z[:, np.newaxis]
    return p_c_z


def compute_acc_score_unc(z, means, tau):
    distance = np.array(
        [np.linalg.norm(z[np.newaxis, i] - means, axis=1) for i in range(z.shape[0])]
    )
    s = np.min(distance, axis=1)
    unc = -np.abs(s - tau)
    unc = unc - unc.min()
    return unc, distance

In [3]:
def generate_class_points(means, N, variance, class_colors, seed=1):
    rng = np.random.default_rng(seed)
    class_to_points = []
    K = np.array([[variance, 0], [0, variance]])
    point_colors = []
    for i in range(means.shape[0]):
        class_to_points.append(rng.multivariate_normal(mean=means[i], cov=K, size=N))
        point_colors += [class_colors[i]] * N
    return np.concatenate(class_to_points, axis=0), point_colors

In [4]:
def add_image(fig, image, x, y, size):
    fig = fig.add_layout_image(
        dict(
            source=image,
            xref="x",
            yref="y",
            x=-1.26,
            y=2.55,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )
    return fig

In [13]:
# Параметры
uncertanty_model = "galue"  #'acc_score' #'galue'
figure_name = "entropy"  #'scf_unc'
means = np.array([[-3, 0], [3, 0], [-1.8, 1.8]])  # , [2, 4.3]])
out_of_gallery_points = np.array([[1.26, 4.0], [0.14, -1.95], [5.35, 3.36]])
class_colors = [
    "#7f7f7f",  #'tab:gray', # see https://stackoverflow.com/questions/64369710/what-are-the-hex-codes-of-matplotlib-tab10-palette
    "#8c564b",  #'tab:brown',
    "#ff7f0e",  #  'tab:green',
    # "#e377c2",  #'tab:pink'
]
marker_color = "#d62728"
marker_size = 14
point_size = 10
inserted_image_size = 1.2


false_accept_point = [-2.428, -1.054]
false_identification = [-2.508, 0.719]
false_reject = [-0.77, -3.70]  # [-0.1, -3.36]
class_labels = np.arange(means.shape[0])
size = 300
value_range = 6
kappa = 1
x_shift = 0
y_shift = 1

beta = 0.5
K = means.shape[0]
M = value_range * 2
tau = (
    -np.log((beta / (M**2)) / (((1 - beta) / K) * (kappa / (2 * np.pi)))) / kappa
) ** 0.5

# Сетка
x = np.linspace(-value_range, value_range, size)
y = np.linspace(-value_range + y_shift, value_range - y_shift, size)
grid_x, grid_y = np.meshgrid(x, y, indexing="ij")
grid_product = np.array(list(product(x, y)))


# Вычисление вероятностей и неопределенности
if uncertanty_model == "galue":
    p_c_z = compute_class_probs(grid_product, means, kappa, value_range * 2)
    uncertainty = -np.sum(
        p_c_z * np.log(p_c_z + 1e-15), axis=1
    )  # Добавлен эпсилон против log(0)
elif uncertanty_model == "acc_score":
    uncertainty, distance = compute_acc_score_unc(grid_product, means, tau)

z = uncertainty.reshape((size, size))

# Создание фигуры
fig = go.Figure()

# Контурный график неопределенности
draw_unc = True
if draw_unc:
    fig.add_trace(
        go.Contour(
            x=x,
            y=y,
            z=z.T,
            colorscale="Blues",  # аналог cm.Blues
            opacity=1,
            contours=dict(
                showlabels=False,
                coloring="fill",
                start=z.min(),  # начальное значение
                end=z.max(),  # конечное значение
                size=(z.max() - z.min()) / 20,
            ),
            line=dict(width=0),
            showscale=False,
            # hoverinfo='skip',
        )
    )

# false reject embedding distribution
draw_scf = False
if uncertanty_model == "galue" and draw_scf:
    rv = multivariate_normal([-0.77, -3.70], [[0.25, 0], [0, 0.25]])
    if draw_unc:
        size = 100
        grid_x_embedding = np.linspace(-2, 0.6, size)
        grid_y_embedding = np.linspace(-2.29, -5, size)
    else:
        size = 300
        grid_x_embedding = x
        grid_y_embedding = y
    grid_product_embedding_distr = np.array(
        list(product(grid_x_embedding, grid_y_embedding))
    )
    embedding_dencity = rv.pdf(grid_product_embedding_distr)
    embedding_dencity = embedding_dencity.reshape((size, size))

    fig.add_trace(
        go.Contour(
            x=grid_x_embedding,
            y=grid_y_embedding,
            z=embedding_dencity.T,
            colorscale="BuGn",
            opacity=0.8,
            contours=dict(
                showlabels=False,
                coloring="fill",
                start=embedding_dencity.min(),  # начальное значение
                end=embedding_dencity.max(),  # конечное значение
                size=(embedding_dencity.max() - embedding_dencity.min()) / 8,
            ),
            line=dict(width=0),
            showscale=False,
            # hoverinfo='skip',
        )
    )


# compute decision boundary

eps = 0
draw_boundary = True
if uncertanty_model == "galue" and draw_boundary:
    for i in range(means.shape[0]):
        other_classes = np.concatenate([p_c_z[:, :i], p_c_z[:, i + 1 :]], axis=1)
        boundary_class = p_c_z[:, i] - np.max(other_classes, axis=1)
        boundary_class = boundary_class.reshape((size, size))

        fig.add_trace(
            go.Contour(
                x=x,
                y=y,
                z=boundary_class.T,
                contours=dict(
                    coloring="lines",  # только линии, без заливки
                    start=0,
                    end=0,
                    size=eps,  # один уровень
                ),
                line=dict(width=2, color="white", dash="longdash"),
                showscale=False,
                showlegend=True,
                hoverinfo="skip",
            )
        )
elif uncertanty_model == "acc_score" and draw_boundary:
    boundary_class = np.min(distance, axis=1) - tau
    boundary_class = boundary_class.reshape((size, size))
    fig.add_trace(
        go.Contour(
            x=x,
            y=y,
            z=boundary_class.T,
            contours=dict(
                coloring="lines",  # только линии, без заливки
                start=0,
                end=0,
                size=eps,  # один уровень
            ),
            line=dict(width=2, color="white", dash="longdash"),
            showscale=False,
            showlegend=True,
            hoverinfo="skip",
        )
    )


# Точки центров классов
fig.add_trace(
    go.Scatter(
        x=means[:, 0],
        y=means[:, 1],
        mode="markers",
        marker=dict(
            size=12, color=class_colors, line=dict(width=1, color="DarkSlateGrey")
        ),
        name="Class Centers",
        hoverinfo="skip",
    )
)

# class samples
class_to_points, point_colors = generate_class_points(
    means, 3, 0.1, class_colors, seed=4
)
fig.add_trace(
    go.Scatter(
        x=class_to_points[:, 0],
        y=class_to_points[:, 1],
        mode="markers",
        marker=dict(
            size=point_size,
            color=point_colors,
            line=dict(width=1, color="DarkSlateGrey"),
            symbol="square",
        ),
        name="class samples",
        hoverinfo="skip",
    )
)
# out of gallery samples

fig.add_trace(
    go.Scatter(
        x=out_of_gallery_points[:, 0],
        y=out_of_gallery_points[:, 1],
        mode="markers",
        marker=dict(
            size=point_size,
            color="#e377c2",
            line=dict(width=1, color="DarkSlateGrey"),
            symbol="square",
        ),
        name="class samples",
        hoverinfo="skip",
    )
)


# false acceptance
fig.add_trace(
    go.Scatter(
        x=[false_accept_point[0]],
        y=[false_accept_point[1]],
        mode="markers",
        marker=dict(size=marker_size, color="#e377c2", symbol="star"),
        name="False Accept Point",
    )
)

# false identification
fig.add_trace(
    go.Scatter(
        x=[false_identification[0]],
        y=[false_identification[1]],
        mode="markers",
        marker=dict(size=marker_size, color="#ff7f0e", symbol="diamond"),
        name="False Indent Point",
    )
)

# false reject
fig.add_trace(
    go.Scatter(
        x=[false_reject[0]],
        y=[false_reject[1]],
        mode="markers",
        marker=dict(size=marker_size, color="#8c564b", symbol="x"),
        name="False Reject Point",
    )
)
add_images = False
if add_images:
    # add images
    false_ident_image = Image.open("plot_images/false_ident.jpg")
    false_accept_image = Image.open("plot_images/fox_1.png")

    class_image3_1 = Image.open("plot_images/pug.jpg")
    class_image3_2 = Image.open("plot_images/pug2.jpg")

    class_image1_1 = Image.open("plot_images/corgi_1.jpg")
    class_image1_2 = Image.open("plot_images/corgi_2.jpg")

    class_image4_1 = Image.open("plot_images/cat_1.jpg")
    class_image4_2 = Image.open("plot_images/cat_2.jpg")
    class_image4_3 = Image.open("plot_images/blurry_cat.jpg")

    # Преобразуем в NumPy array
    # img_array = np.array(image)
    false_ident_image_pos = [-4.4, 3.7]

    fig.add_annotation(
        x=false_identification[0],
        y=false_identification[1],  # Arrowhead position
        xref="x",
        yref="y",
        text="",  # No text for a pure arrow
        showarrow=True,
        arrowhead=1,  # Arrowhead style
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="#7f7f7f",
        ax=-4.4,
        ay=3.0936,  # Arrow tail position
        axref="x",
        ayref="y",
    )
    # fig = add_image(fig, false_ident_image, false_ident_image_pos[0], false_ident_image_pos[1], inserted_image_size)
    fig.add_layout_image(
        dict(
            source=false_ident_image,
            xref="x",
            yref="y",
            x=false_ident_image_pos[0],
            y=false_ident_image_pos[1],
            sizex=inserted_image_size,
            sizey=inserted_image_size,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )
    fig.add_layout_image(
        dict(
            source=false_accept_image,
            xref="x",
            yref="y",
            x=-3,
            y=-1.75,
            sizex=inserted_image_size,
            sizey=inserted_image_size,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )

    fig.add_layout_image(
        dict(
            source=class_image3_1,
            xref="x",
            yref="y",
            x=-1.26,
            y=2.55,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )
    fig.add_layout_image(
        dict(
            source=class_image3_2,
            xref="x",
            yref="y",
            x=-0.98,
            y=1.52,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )

    fig.add_layout_image(
        dict(
            source=class_image1_1,
            xref="x",
            yref="y",
            x=-2.18,
            y=-0.21,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )
    fig.add_layout_image(
        dict(
            source=class_image1_2,
            xref="x",
            yref="y",
            x=-3.63,
            y=0.58,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )

    # cats
    fig.add_layout_image(
        dict(
            source=class_image4_1,
            xref="x",
            yref="y",
            x=3.8,
            y=0,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )

    fig.add_layout_image(
        dict(
            source=class_image4_2,
            xref="x",
            yref="y",
            x=2.5,
            y=-0.61,
            sizex=0.7,
            sizey=0.7,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )

    fig.add_layout_image(
        dict(
            source=class_image4_3,
            xref="x",
            yref="y",
            x=0.7,
            y=-2.8,
            sizex=1.2,
            sizey=1.2,
            xanchor="center",
            yanchor="middle",
            opacity=1,  # 0.3,
            layer="above",
        )
    )


# Настройки вида
fig.update_layout(
    title=None,
    xaxis_title=None,
    yaxis_title=None,
    xaxis=dict(scaleanchor="y", scaleratio=1, visible=False),
    yaxis=dict(visible=False),
    showlegend=False,  # Убираем легенду, как в оригинале
    width=12 * 60,  # 600,
    height=10 * 60,  # 600,
    margin=dict(l=0, r=0, t=0, b=0),
    paper_bgcolor="rgba(0,0,0,0)",  # Прозрачный фон (опционально)
    plot_bgcolor="rgba(0,0,0,0)",
    hovermode="closest",
)

# Убираем рамку (spines)
fig.update_xaxes(showline=False, zeroline=False)
fig.update_yaxes(showline=False, zeroline=False)

# Сохранение как PNG через kaleido (нужно установить: pip install kaleido)
# fig.show()
fig.write_image(
    f"{uncertanty_model}_{figure_name}.png",
    format="png",
    width=12 * 60,
    height=10 * 60,
    scale=5,
)  # dpi ~300 при scale=5

# При необходимости: сохранение в PDF
# fig.write_image(f"{uncertanty_model}_{figure_name}.pdf", format="pdf", width=12 * 60, height=10 * 60)
fig.write_image(
    f"{uncertanty_model}_{figure_name}.svg", format="svg", width=12 * 60, height=10 * 60
)

# Отображение (в Jupyter или интерактивной среде)
fig  # .show()

In [ ]:
false_ident_image_pos[0]

Figure 1: A toy example in two-dimensional space showcasing the importance of the information about embeddings relative position. The gallery consists of four classes. Colored circles represent class centers and squares represent samples of corresponding class. Dashed line shows the desicion boundary between rejection and acceptance. Intensity of the blue color is proportional to the enthropy of class distribution given an embedding. We see that in our model the uncertainty is high at the desision boundary and between different classes. The $\blacklozenge$ point lies between cyan and green classes and show the example of high uncertainty due to possibility of misidentification. The $\bigstar$ point has high uncertainty because it is at the desision boundary and could be a false accepted sample. The $\boldsymbol{\times}$ point is false rejected example, we can detect such cases using sample quality estimate. 

In [ ]:
import kaleido

print(kaleido.version)

In [ ]:
np.sum(p_c_z, axis=1).max()